In [ ]:
import numpy as np
import pandas as pd
import pyreadr

In [ ]:
data = pyreadr.read_r("../models/r3PG/vignettes_build/vignette_data/solling.rda")
data.keys()

In [ ]:
climate = pd.DataFrame(data["climate_solling"])
climate["idx"] = np.arange(0, len(climate))

obv = pd.DataFrame(data["observ_solling"])
obv["month"] = pd.to_datetime(obv["date"]).dt.month
obv["year"] = pd.to_datetime(obv["date"]).dt.year

obv = obv.merge(climate, on=["month", "year"])[
    [
        "idx",
        "month",
        "year",
        "date",
        "biom_stem",
        "biom_foliage",
        "biom_root",
        "basal_area",
        "stems_n",
        "dbh",
        "height",
    ]
]

obv = obv.rename(
    columns={
        "dbh": "DBH",
        "biom_stem": "WS",
        "biom_foliage": "WF",
        "biom_root": "WR",
        "basal_area": "BA",
        "stems_n": "N",
        "height": "Height",
    }
)

In [ ]:
param_df = pd.DataFrame(data["param_solling"])

params_bounds = {}
for _, row in param_df.iterrows():
    if pd.notna(row["min"]) and pd.notna(row["max"]):
        params_bounds[row["param_name"]] = (row["min"], row["max"])

In [ ]:
param_default = pd.read_excel("../data/data.default.xlsx")

params = pd.DataFrame(data["param_solling"])
params = params.rename(columns={"param_name": "parameter", "default": "piab"})
params = params[["parameter", "piab"]]
params["piab"] = params["piab"].fillna(param_default["default"])

params.head()

In [ ]:
with pd.ExcelWriter("../data/solling_data.xlsx", engine="openpyxl") as writer:
    data["climate_solling"].to_excel(writer, sheet_name="climate", index=False)
    data["site_solling"].to_excel(writer, sheet_name="site", index=False)
    data["species_solling"].to_excel(writer, sheet_name="species", index=False)
    params.to_excel(writer, sheet_name="parameters", index=False)
    pd.DataFrame().to_excel(writer, sheet_name="thinning", index=False)
    pd.DataFrame().to_excel(writer, sheet_name="sizeDist", index=False)
    obv.to_excel(writer, sheet_name="observed", index=False)
    data["param_solling"].to_excel(writer, sheet_name="param_bound", index=False)